# Blue Resistors — SAM 3.1 COCO segmentation

Process **[Blue.Resistors.v1i.coco.zip](https://github.com/Persie0/resistor_model/releases/download/r1/Blue.Resistors.v1i.coco.zip)** with the same SAM 3.1 model and export pipeline as [the existing three-dataset workflow](https://github.com/Persie0/resistor_model/blob/main/colab/sam3_merge_3_resistor_datasets_colab.ipynb).

**Run these cells in order on a Colab GPU runtime.** There is no manual image upload: the dataset ZIP is downloaded automatically, SHA-256 verified, and source COCO JSON files are used only to enumerate images. The original labels and segmentations are ignored.

SAM detects the **blue axial resistor body**, selects **one highest-scoring nonempty mask per image**, and keeps masks at **60% confidence or higher** by default. Output: COCO RLE annotations, masks, overlay previews, a QA manifest, and an 80/10/10 train/valid/test split.

Google Drive checkpoints are stored independently at `MyDrive/resistor_sam3_blue_checkpoint/` every 25 processed images. Rerun after disconnecting to resume. The final `blue_resistors_sam3.zip` is saved to that Drive folder and downloaded to your browser.

SAM 3.1 weights are retrieved from the public `AEmotionStudio/sam3.1` mirror (~3.5 GB) by the shared bootstrap. Inference requires a GPU.

## 1. Load SAM 3.1 (GPU required)

In [ ]:
import urllib.request
bootstrap_url = "https://raw.githubusercontent.com/Persie0/resistor_model/main/colab/sam3_test_bootstrap.py"
bootstrap_path = "/content/sam3_test_bootstrap.py"
if not globals().get("SAM3_BOOTSTRAPPED", False):
    urllib.request.urlretrieve(bootstrap_url, bootstrap_path)
    exec(compile(open(bootstrap_path, encoding="utf-8").read(), bootstrap_path, "exec"), globals(), globals())
else:
    print("[sam3] reusing the loaded SAM 3.1 image model")

## 2. Run SAM 3.1 on the entire Blue Resistors dataset

Set the prompt and acceptance threshold below if needed. Changing either automatically invalidates the previous inference checkpoint (because the parameters are part of the checkpoint identity).

In [ ]:
import os
import urllib.request

os.environ["SAM3_BLUE_PROMPT"] = "blue axial resistor body"
os.environ["SAM3_BLUE_MIN_SCORE"] = "0.60"

workflow_url = "https://raw.githubusercontent.com/Persie0/resistor_model/main/colab/sam3_blue_resistors.py"
workflow_path = "/content/sam3_blue_resistors.py"
urllib.request.urlretrieve(workflow_url, workflow_path)
exec(compile(open(workflow_path, encoding="utf-8").read(), workflow_path, "exec"), globals(), globals())

## 3. Preview generated masks

View examples from the exported QA overlay images. The full ZIP includes every exported mask and its matching COCO annotation.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

overlays = sorted(Path("/content/blue_resistors_sam3").glob("*/overlays/*.jpg"))
if not overlays:
    print("No QA overlays found. Run the dataset processing cell first.")
else:
    sample = overlays[:min(6, len(overlays))]
    fig, axes = plt.subplots(1, len(sample), figsize=(min(24, 4 * len(sample)), 5))
    if len(sample) == 1:
        axes = [axes]
    for axis, path in zip(axes, sample):
        with Image.open(path) as img:
            axis.imshow(img)
        axis.set_title(f"{path.parent.parent.name} / {path.stem}", fontsize=9)
        axis.axis("off")
    plt.tight_layout()
    plt.show()